# Data — analyzer

**Block 3 of 3 in the Data stage.** The other two blocks *produce* data; this notebook is where it
is *understood*, and where a feature either earns a backtest or is dropped before anyone spends a
week on it.

```
Data/curator.py    + Data/Curator/custom_calculations.py    ->  Curator/Time_Series/   m_* + c_*
Data/refinery.py   + Data/Refinery/custom_calculations.py   ->  Refinery/Time_Series/  + r_*
Data/analyzer.ipynb                                          ->  Analyzer/Charts/, the IC table
```

| Column family | Built by | Scope |
| --- | --- | --- |
| `m_*` | the provider, via the Curator | raw market data |
| `c_*` | `Curator/custom_calculations.py` | **per security** — one asset's own history |
| `r_*` | `Refinery/custom_calculations.py` | **cross-sectional**, or fitted — see that module |

## What this notebook is for

`Universe/universe.ipynb` profiles the *catalogue*: what exists, what is missing, when each asset
becomes usable. This notebook looks at the **content** — what the data says, and whether the
signal built on it carries anything.

It ends in two places a strategy has to pass through:

- **Section 4** measures whether the regime signal separates anything at all, and **section 5
  measures how much of any separation is look-ahead.** That second number is the one worth the
  price of admission.
- **Section 8** is the information coefficient table: for every candidate feature, the
  cross-sectional correlation with forward returns. **A feature that fails here does not get a
  book built on it** — and one that passes has earned a backtest, not a belief.

> **Write down what you expect before you run this.** A prediction made from the data and then
> confirmed by the engine is the strongest methodological result an experiment can report. A
> number found first and explained afterwards is a story.

---

## 0 · Setup

**Four names in this cell belong to the strategy. Everything else in the notebook is process.**
Point them at your own columns and every section below re-runs unchanged.

In [ ]:
"""Step 3 - Data analyzer. Exploratory analysis over the Curator and Refinery output."""
import pathlib

# --- example: begin ---
import sys

# --- example: end ---
import matplotlib.colors
import matplotlib.pyplot
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory holding pyproject.toml and Data/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Data").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
CURATOR_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"
REFINERY_DIR = REPO_ROOT / "Data" / "Refinery" / "Time_Series"
CHART_DIR = REPO_ROOT / "Data" / "Analyzer" / "Charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# --- example: begin ---
# The regime model lives in the Refinery; sections 5 and 6 re-run it to compare variants.
sys.path.insert(0, str(REPO_ROOT / "Data" / "Refinery"))
import jump_model

# --- example: end ---

SECURITY_MASTER = pandas.read_csv(REPO_ROOT / "Universe" / "Security_Master.csv")

# --- The strategy's columns: the only names here that are not process -----------------------
# --- example: begin ---
ELIGIBILITY_COLUMN = "r_regime_bull"   # 1.0 when an asset may be held; None until you have one
REGIME_COLUMN = "r_regime"             # a fitted per-security state, or None
FEATURE_COLUMNS = [                    # what a fitted model eats, in the Refinery's order
    "c_return_ewm_hl5", "c_return_ewm_hl10", "c_return_ewm_hl21",
    "c_downside_deviation_log_hl5", "c_downside_deviation_log_hl21",
    "c_sortino_hl5", "c_sortino_hl10", "c_sortino_hl21",
]
CANDIDATE_RANK_COLUMNS = [             # cross-sectional candidates the IC table screens
    "r_return_ewm_hl21_rank",
    "r_sortino_hl21_rank",
    "r_downside_deviation_hl21_rank",
    "r_liquidity_rank",
]
EXTRA_PANEL_COLUMNS = ["r_regime_bull_breadth"]
# --- example: end ---

PANEL_COLUMNS = [
    "m_date",
    "m_close_dividend_and_split_adjusted",
    "c_return_1d",
    "c_daily_traded_value_63d",
    "r_universe_size",
    *([REGIME_COLUMN] if REGIME_COLUMN else []),
    *([ELIGIBILITY_COLUMN] if ELIGIBILITY_COLUMN else []),
    *EXTRA_PANEL_COLUMNS,
    *FEATURE_COLUMNS,
    *CANDIDATE_RANK_COLUMNS,
]

refinery_paths = sorted(REFINERY_DIR.glob("*.csv"))
assert refinery_paths, f"no refined files in {REFINERY_DIR} - run: uv run python Data/refinery.py"

frames = []
for path in refinery_paths:
    frame = pandas.read_csv(path, usecols=PANEL_COLUMNS, parse_dates=["m_date"])
    frame.insert(0, "ticker", path.stem)
    frames.append(frame)

panel = pandas.concat(frames, ignore_index=True).sort_values(["m_date", "ticker"])

# Ordered so every chart lists assets the same way: by group, then by class within it. The
# classification comes from the security master rather than from the panel because it is a
# property of the catalogue, not of a date - which is exactly why the refinery suffixes the
# joined copy `_current`.
ASSET_ORDER = SECURITY_MASTER.sort_values(["asset_group", "asset_class"])["ticker"].tolist()
CLASSIFICATION = SECURITY_MASTER.set_index("ticker")[["asset_class", "asset_group"]]

print(f"Refined files : {len(refinery_paths)}")
print(f"Panel         : {panel['ticker'].nunique()} assets x {panel['m_date'].nunique()} dates"
      f" = {len(panel):,} rows")
print(f"Priced        : {panel['m_date'].min().date()} -> {panel['m_date'].max().date()}")
# --- example: begin ---
signalled = panel.dropna(subset=[REGIME_COLUMN])
print(f"Signalled     : {signalled['m_date'].min().date()}"
      f" -> {signalled['m_date'].max().date()}"
      f"  ({len(signalled) / len(panel):.0%} of rows)")
# --- example: end ---

In [ ]:
INK = "#0b0b0b"
MUTED = "#52514e"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREEN = "#2f9e6b"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": MUTED, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 12, "figure.dpi": 110,
    "savefig.dpi": 160, "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, xlabel=None, ylabel=None):
    """Left-aligned bold title, optional subtitle line, recessive grid, no top/right spines."""
    if title:
        axes.set_title(title, loc="left", pad=24 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(subtitle, xy=(0, 1), xycoords="axes fraction", xytext=(0, 6),
                      textcoords="offset points", fontsize=9, color=MUTED, va="bottom", ha="left")
    if xlabel:
        axes.set_xlabel(xlabel)
    if ylabel:
        axes.set_ylabel(ylabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)
    return axes


def save(figure, file_name):
    """Write a figure to Data/Analyzer/Charts/ and show it."""
    figure.tight_layout()
    figure.savefig(CHART_DIR / file_name)
    matplotlib.pyplot.show()


print("Chart helpers ready.")

---

## 1 · What each stage contributed

The refined file is the Curator file plus columns, same rows. Confirming that here is what lets
every section below read one directory and forget the Curator exists.

In [ ]:
sample = refinery_paths[0].stem
curator_columns = list(pandas.read_csv(CURATOR_DIR / f"{sample}.csv", nrows=0).columns)
refinery_columns = list(pandas.read_csv(REFINERY_DIR / f"{sample}.csv", nrows=0).columns)

dropped = [column for column in curator_columns if column not in refinery_columns]
added = [column for column in refinery_columns if column not in curator_columns]

print(pandas.DataFrame({
    "family": ["m_* (provider)", "c_* (Curator)", "r_* (Refinery)"],
    "columns": [
        len([c for c in refinery_columns if c.startswith("m_")]),
        len([c for c in refinery_columns if c.startswith("c_")]),
        len([c for c in refinery_columns if c.startswith("r_")]),
    ],
}).to_string(index=False))
print(f"\nDropped by the refinery: {dropped or 'none - the Curator output is carried through'}")
print(f"Added ({len(added)}): {', '.join(added)}")

profiled = FEATURE_COLUMNS + CANDIDATE_RANK_COLUMNS + ([REGIME_COLUMN] if REGIME_COLUMN else [])
coverage = panel[profiled].notna().mean()
print("\nColumn coverage across the panel:")
print(coverage.map("{:.1%}".format).to_string())
# --- example: begin ---
print("\n-> the regime column is short by design: it is null through the model's warm-up window.")
# --- example: end ---

---

## 2 · What diversification is actually available

A multi-asset strategy is a bet that the assets do not all move together. Whether that is true is
measurable, and it decides how much the strategy can possibly add: if everything is one trade,
choosing between them is theatre.

In [ ]:
returns_wide = panel.pivot_table(
    index="m_date", columns="ticker", values="c_return_1d", aggfunc="first"
)[ASSET_ORDER]

annual = pandas.DataFrame({
    "return": returns_wide.mean() * 252,
    "volatility": returns_wide.std() * numpy.sqrt(252),
})
annual["sharpe"] = annual["return"] / annual["volatility"]
annual["worst_day"] = returns_wide.min()
annual = annual.join(CLASSIFICATION)
print("Buy and hold, whole priced window:")
print(annual.round(3).to_string())

correlation = returns_wide.corr()
figure, axes = matplotlib.pyplot.subplots(figsize=(8.4, 7))
mesh = axes.imshow(correlation.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes.set_xticks(range(len(ASSET_ORDER)))
axes.set_xticklabels(ASSET_ORDER, rotation=90)
axes.set_yticks(range(len(ASSET_ORDER)))
axes.set_yticklabels(ASSET_ORDER)
for row in range(len(ASSET_ORDER)):
    for column in range(len(ASSET_ORDER)):
        value = correlation.iat[row, column]
        axes.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=7,
                  color="white" if abs(value) > 0.6 else INK)
axes.set_title("Daily return correlation", loc="left", weight="bold")
figure.colorbar(mesh, ax=axes, shrink=0.72)
save(figure, "asset_correlation.png")

off_diagonal = correlation.to_numpy()[~numpy.eye(len(ASSET_ORDER), dtype=bool)]
print(f"\nMean off-diagonal correlation: {off_diagonal.mean():.2f}")
print(f"Highest pair : {off_diagonal.max():.2f}")
print(f"Lowest pair  : {off_diagonal.min():.2f}")
print("\n-> the lower this is, the more a strategy that chooses between them can add.")

---

<!-- EXAMPLE-ONLY CELL -->
## 3 · The regime signal, seen

One row per asset, one column per day, blue where the model says the asset is in its good state.
This is the whole signal on one screen, and it is worth looking at before any statistic: regimes
that line up vertically across assets are a market-wide event the strategy cannot diversify away,
and regimes that alternate are the thing it is meant to exploit.

In [ ]:
# EXAMPLE-ONLY CELL
regime_wide = panel.pivot_table(
    index="m_date", columns="ticker", values=ELIGIBILITY_COLUMN, aggfunc="first"
)[ASSET_ORDER].dropna(how="all")

figure, axes = matplotlib.pyplot.subplots(figsize=(13, 4.8))
axes.pcolormesh(
    regime_wide.index,
    numpy.arange(len(ASSET_ORDER)),
    numpy.ma.masked_invalid(regime_wide.to_numpy().T),
    cmap=matplotlib.colors.ListedColormap([ORANGE, BLUE]),
    vmin=0, vmax=1, shading="nearest",
)
axes.set_yticks(numpy.arange(len(ASSET_ORDER)))
axes.set_yticklabels(ASSET_ORDER)
axes.set_title("Regime by asset: blue is the good state, orange the bad one",
               loc="left", weight="bold", pad=24)
axes.annotate("blank means the model had not been fitted for that asset yet", xy=(0, 1),
              xycoords="axes fraction", xytext=(0, 6), textcoords="offset points",
              fontsize=9, color=MUTED, va="bottom", ha="left")
save(figure, "regime_timeline.png")

<!-- EXAMPLE-ONLY CELL -->
### 3.1 · Persistence — what the jump penalty bought

A regime model with no penalty is a volatility indicator that changes its mind every few days, and
a strategy built on it pays transaction costs for noise. The two numbers that matter are how often
the label flips and how long a spell lasts; together they set the floor on turnover **before** any
portfolio rule is applied.

In [ ]:
# EXAMPLE-ONLY CELL
signalled = panel.dropna(subset=[REGIME_COLUMN]).sort_values(["ticker", "m_date"])
rows = []
spell_lengths = []
for ticker, group in signalled.groupby("ticker"):
    spells = (group[REGIME_COLUMN] != group[REGIME_COLUMN].shift()).cumsum()
    lengths = group.groupby(spells).size()
    spell_lengths.extend(lengths.tolist())
    years = len(group) / 252
    rows.append({
        "ticker": ticker,
        "days": len(group),
        "bull_share": (group[ELIGIBILITY_COLUMN] == 1.0).mean(),
        "switches_per_year": (len(lengths) - 1) / years,
        "median_spell_days": int(lengths.median()),
        "shortest_spell": int(lengths.min()),
    })

persistence = pandas.DataFrame(rows).set_index("ticker").loc[ASSET_ORDER]
print(persistence.round(2).to_string())
print(f"\nAcross all assets: {numpy.median(spell_lengths):.0f} trading days median spell,"
      f" {persistence['switches_per_year'].mean():.1f} switches per year.")

figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist(numpy.clip(spell_lengths, 0, 400), bins=50, color=BLUE)
axes[0].axvline(numpy.median(spell_lengths), color=ORANGE, linewidth=1.6)
style_axes(axes[0], "How long a regime lasts",
           subtitle=f"median {numpy.median(spell_lengths):.0f} trading days (orange);"
                    " clipped at 400 for display",
           xlabel="consecutive trading days in one regime", ylabel="spells")

breadth = panel.groupby("m_date")["r_regime_bull_breadth"].first().dropna()
axes[1].fill_between(breadth.index, breadth.to_numpy(), color=BLUE, alpha=0.22)
axes[1].plot(breadth.index, breadth.to_numpy(), color=BLUE, linewidth=1.2)
axes[1].axhline(0.5, color=MUTED, linewidth=1, linestyle="--")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
style_axes(axes[1], "Breadth: share of assets in their good regime",
           subtitle="the one reading that is about the market rather than about an asset",
           ylabel="share of assets")
save(figure, "regime_persistence_and_breadth.png")

print(f"\nBreadth: mean {breadth.mean():.0%}, and it spends"
      f" {(breadth <= 0.25).mean():.1%} of days at or below 25%.")

---

<!-- EXAMPLE-ONLY CELL -->
## 4 · Does the regime separate anything?

The decisive question, and it has to be asked on **forward** returns. Splitting today's return by
today's label proves nothing — a model that reads a bad day as a bad regime is describing, not
predicting. What matters is the return earned *after* the label was struck.

Two things are being measured at once, and they can come out differently:

- **forward return** — does the good regime earn more?
- **forward volatility** — is the good regime calmer?

A signal can be worth trading on the second alone. It would not be the first.

In [ ]:
# EXAMPLE-ONLY CELL
FORWARD_HORIZON_DAYS = 1

ordered = panel.sort_values(["ticker", "m_date"]).copy()
ordered["forward_return"] = (
    ordered.groupby("ticker")["c_return_1d"].shift(-FORWARD_HORIZON_DAYS)
)
measurable = ordered.dropna(subset=[ELIGIBILITY_COLUMN, "forward_return"])

rows = []
for ticker, group in measurable.groupby("ticker"):
    bull = group[ELIGIBILITY_COLUMN] == 1.0
    rows.append({
        "ticker": ticker,
        "bull_share": bull.mean(),
        "return_bull": group.loc[bull, "forward_return"].mean() * 252,
        "return_bear": group.loc[~bull, "forward_return"].mean() * 252,
        "volatility_bull": group.loc[bull, "forward_return"].std() * numpy.sqrt(252),
        "volatility_bear": group.loc[~bull, "forward_return"].std() * numpy.sqrt(252),
    })

separation = pandas.DataFrame(rows).set_index("ticker").loc[ASSET_ORDER]
separation["return_spread"] = separation["return_bull"] - separation["return_bear"]
separation["volatility_drop"] = separation["volatility_bear"] - separation["volatility_bull"]
print("Annualised, measured on the day AFTER the label:")
print(separation.round(3).to_string())

return_wins = int((separation["return_spread"] > 0).sum())
volatility_wins = int((separation["volatility_drop"] > 0).sum())
print(f"\nGood regime earns more   : {return_wins}/{len(separation)} assets,"
      f" mean spread {separation['return_spread'].mean():+.1%}")
print(f"Good regime is calmer    : {volatility_wins}/{len(separation)} assets,"
      f" mean drop   {separation['volatility_drop'].mean():+.1%}")

figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.6))
positions = numpy.arange(len(separation))
for axis, column, title, subtitle in (
    (axes[0], "return_spread", "Forward return: good regime minus bad",
     "positive means the label predicts return"),
    (axes[1], "volatility_drop", "Forward volatility: bad regime minus good",
     "positive means the label predicts calm"),
):
    values = separation[column].to_numpy()
    axis.barh(positions, values, color=[GREEN if v > 0 else ORANGE for v in values], height=0.7)
    axis.set_yticks(positions)
    axis.set_yticklabels(separation.index)
    axis.axvline(0, color=INK, linewidth=1.2)
    axis.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    style_axes(axis, title, subtitle=subtitle, xlabel="annualised")
    axis.spines["left"].set_visible(False)
save(figure, "regime_separation.png")

---

<!-- EXAMPLE-ONLY CELL -->
## 5 · The number this notebook exists for

The section above used the **causal** label: the regime the model assigns to a day using only
that day and everything before it. There is a second label available from the same model — the
**smoothed** one, which is the best explanation of a finished stretch of history and is allowed to
know what happened next.

The two look nearly identical on a chart. They differ exactly at the turning points, which is
where every dollar is. Fitting the model once over the whole history and reading its smoothed
labels is a plausible-looking mistake, and it takes one line of code.

**Below is what that line is worth, in annualised return, measured on this data.** Whatever the
number turns out to be, it is the size of the prize for getting look-ahead wrong here — and the
reason `Data/Refinery/jump_model.py` returns the causal label and says so in its docstring.

In [ ]:
# EXAMPLE-ONLY CELL
def smoothed_labels_for(frame):
    """
    Fit one model on an asset's whole history and read its smoothed labels back.

    This is the look-ahead version, built deliberately so its cost can be measured. Nothing in
    the pipeline calls it.
    """
    features = frame[FEATURE_COLUMNS].to_numpy(dtype="float64")
    returns = frame["c_return_1d"].to_numpy(dtype="float64")
    usable = numpy.flatnonzero(numpy.isfinite(features).all(axis=1) & numpy.isfinite(returns))
    model = jump_model.fit_regime_model(
        features[usable], returns[usable],
        regimes_count=2, jump_penalty=50.0, clipping_deviations=3.0,
    )
    scaled = (
        numpy.clip(features[usable], model.lower_bound, model.upper_bound) - model.location
    ) / model.scale
    losses = 0.5 * (
        (scaled[:, numpy.newaxis, :] - model.centers[numpy.newaxis, :, :]) ** 2
    ).sum(axis=2)
    labels = pandas.Series(numpy.nan, index=frame.index, dtype="float64")
    labels.iloc[usable] = jump_model._smoothed_labels(losses, 50.0)
    return labels


rows = []
for ticker, group in ordered.groupby("ticker"):
    group = group.copy()
    smoothed = smoothed_labels_for(group)
    group["smoothed_bull"] = (
        (smoothed == jump_model.REGIME_BULL).astype("float64").where(smoothed.notna())
    )
    usable = group.dropna(subset=["forward_return", "smoothed_bull", ELIGIBILITY_COLUMN])
    rows.append({
        "ticker": ticker,
        "causal": (
            usable.loc[usable[ELIGIBILITY_COLUMN] == 1.0, "forward_return"].mean() * 252
            - usable.loc[usable[ELIGIBILITY_COLUMN] == 0.0, "forward_return"].mean() * 252
        ),
        "smoothed": (
            usable.loc[usable["smoothed_bull"] == 1.0, "forward_return"].mean() * 252
            - usable.loc[usable["smoothed_bull"] == 0.0, "forward_return"].mean() * 252
        ),
        "agreement": (usable["smoothed_bull"] == usable[ELIGIBILITY_COLUMN]).mean(),
    })

lookahead = pandas.DataFrame(rows).set_index("ticker").loc[ASSET_ORDER]
lookahead["cost_of_honesty"] = lookahead["smoothed"] - lookahead["causal"]
print("Good-minus-bad forward return spread, annualised:")
print(lookahead.round(3).to_string())
print(f"\nThe two labels agree on {lookahead['agreement'].mean():.0%} of days.")
print(f"Smoothed (uses the future) : {lookahead['smoothed'].mean():+.1%} mean spread,"
      f" {int((lookahead['smoothed'] > 0).sum())}/{len(lookahead)} assets positive")
print(f"Causal   (tradable)        : {lookahead['causal'].mean():+.1%} mean spread,"
      f" {int((lookahead['causal'] > 0).sum())}/{len(lookahead)} assets positive")
print(f"\nLOOK-AHEAD IS WORTH {lookahead['cost_of_honesty'].mean():.1%} A YEAR ON THIS DATA.")

figure, axes = matplotlib.pyplot.subplots(figsize=(11, 4.8))
positions = numpy.arange(len(lookahead))
axes.barh(positions - 0.2, lookahead["smoothed"], height=0.38, color=ORANGE,
          label="smoothed - uses the future")
axes.barh(positions + 0.2, lookahead["causal"], height=0.38, color=BLUE,
          label="causal - tradable")
axes.set_yticks(positions)
axes.set_yticklabels(lookahead.index)
axes.axvline(0, color=INK, linewidth=1.2)
axes.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes.legend(frameon=False, fontsize=9)
style_axes(axes, "The same model, read two ways",
           subtitle="good-minus-bad forward return spread; the gap between the bars is look-ahead",
           xlabel="annualised spread")
axes.spines["left"].set_visible(False)
save(figure, "lookahead_cost.png")

---

<!-- EXAMPLE-ONLY CELL -->
## 6 · The jump penalty, read as a curve

The penalty is the model's one real knob. **Read this as a curve, not a cell**: a parameter that
degrades monotonically across settings is information, and a setting that wins by a hair over its
neighbours is noise.

The honest way to choose it is on a property of the *signal* — how often it flips, how long spells
last — and not on return, because choosing it on return is the data-snooping the whole process is
built to prevent. What this table is for is showing whether the conclusion in sections 4 and 5
survives every setting, or only the one that was picked.

> Roughly a minute: it refits every asset at every penalty.

In [ ]:
# EXAMPLE-ONLY CELL
PENALTY_GRID = (5.0, 15.0, 30.0, 50.0, 100.0)
RUN_PENALTY_SWEEP = True

curator_frames = {
    path.stem: pandas.read_csv(path, parse_dates=["m_date"]).sort_values("m_date")
    for path in sorted(CURATOR_DIR.glob("*.csv"))
    if path.stem in set(ASSET_ORDER)
}

sweep_rows = []
if RUN_PENALTY_SWEEP:
    for penalty in PENALTY_GRID:
        for ticker, frame in curator_frames.items():
            labels = jump_model.rolling_regime_labels(
                frame[FEATURE_COLUMNS], frame["c_return_1d"],
                training_days=1260, refit_every_days=126,
                regimes_count=2, jump_penalty=penalty, clipping_deviations=3.0,
            )
            forward = frame["c_return_1d"].shift(-1)
            usable = labels.notna() & forward.notna()
            bull = usable & (labels == jump_model.REGIME_BULL)
            bear = usable & (labels != jump_model.REGIME_BULL)
            sweep_rows.append({
                "penalty": penalty, "ticker": ticker,
                "switches_per_year": (labels.dropna().diff() != 0).sum() / (usable.sum() / 252),
                "return_spread": (forward[bull].mean() - forward[bear].mean()) * 252,
                "volatility_drop": (forward[bear].std() - forward[bull].std()) * numpy.sqrt(252),
            })

sweep = pandas.DataFrame(sweep_rows)
summary = sweep.groupby("penalty").agg(
    switches_per_year=("switches_per_year", "mean"),
    return_spread=("return_spread", "mean"),
    return_positive=("return_spread", lambda values: (values > 0).sum()),
    volatility_drop=("volatility_drop", "mean"),
    volatility_positive=("volatility_drop", lambda values: (values > 0).sum()),
)
print(f"Averaged across {len(ASSET_ORDER)} assets;"
      f" 'positive' counts how many of them the sign holds for.")
print(summary.round(3).to_string())

figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(summary.index, summary["switches_per_year"], marker="o", color=BLUE)
style_axes(axes[0], "Turnover the penalty buys",
           subtitle="regime switches per asset per year", xlabel="jump penalty")
axes[1].plot(summary.index, summary["return_spread"], marker="o", color=ORANGE, label="return")
axes[1].plot(summary.index, summary["volatility_drop"], marker="o", color=GREEN,
             label="volatility drop")
axes[1].axhline(0, color=INK, linewidth=1.2)
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes[1].legend(frameon=False, fontsize=9)
style_axes(axes[1], "What separates, at every setting",
           subtitle="good minus bad, annualised, on the day after the label",
           xlabel="jump penalty")
save(figure, "jump_penalty_sweep.png")

---

## 7 · Are the cross-sectional columns what they claim to be?

The Refinery asserts one property that would be silent if broken: **every rank is a percentile
taken inside a single date.** A rank computed over the pooled sample instead would drift as the
universe's composition changed, and nothing would raise an error.

The check is an identity rather than a rule of thumb. `rank(pct=True)` over *n* untied values
returns `1/n … n/n`, so the mean of a single date's ranks must be exactly `(n + 1) / (2n)` — which
for twelve assets is **0.542, not 0.5**. Testing against 0.5 would look like a small failure on
every date; testing against the identity is exact. On a wide universe the two converge, which is
why the difference is easy to miss and worth writing down here.

In [ ]:
figure, axes_grid = matplotlib.pyplot.subplots(
    1, len(CANDIDATE_RANK_COLUMNS), figsize=(3.4 * len(CANDIDATE_RANK_COLUMNS), 3.4), squeeze=False
)
for axes, column in zip(axes_grid.flatten(), CANDIDATE_RANK_COLUMNS):
    axes.hist(panel[column].dropna(), bins=len(ASSET_ORDER), color=BLUE)
    axes.set_ylim(bottom=0)
    style_axes(axes, column.replace("r_", "").replace("_rank", ""), xlabel="percentile")
figure.suptitle(
    f"Pooled rank distributions - {len(ASSET_ORDER)} even bars is correct",
    x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.9))
figure.savefig(CHART_DIR / "rank_distributions.png")
matplotlib.pyplot.show()

daily_mean = panel.groupby("m_date")[CANDIDATE_RANK_COLUMNS].mean()
daily_count = panel.groupby("m_date")[CANDIDATE_RANK_COLUMNS].count()
expected_mean = (daily_count + 1) / (2 * daily_count)

print("Pooled mean of each per-date rank:")
print(daily_mean.mean().round(4).to_string())
print(f"\nExpected for a full {len(ASSET_ORDER)}-asset cross-section:"
      f" {(len(ASSET_ORDER) + 1) / (2 * len(ASSET_ORDER)):.4f}")
print("\nLargest deviation of any single date's mean from its own (n+1)/2n:")
print((daily_mean - expected_mean).abs().max().round(6).to_string())
print("\n-> deviations at the floating-point level confirm the ranks are per-date, not pooled.")

---

## 8 · Information coefficient — which features predict returns

For each feature and horizon, the **information coefficient** is the cross-sectional Spearman
correlation between the feature's rank on date *t* and the forward return from *t* to *t+h*,
computed **per date** and then averaged. Per date is what keeps it causal: the correlation only
ever compares assets that were observable at the same moment.

- **IC** — the mean daily correlation. The sign matters as much as the size: a negative IC means
  the feature works *inverted*.
- **IR** = IC / IC std — the consistency of the edge, which is what survives into a portfolio.

Computed over the whole panel and, separately, over the **eligible pool** — the assets the regime
model says may be held. The second is the one that matters, because that pool is what the strategy
actually selects from, and a feature can behave differently inside an already-filtered group.

> **On a twelve-asset universe, read these as directional.** A cross-sectional correlation over
> twelve points is noisy; the fundamental law says an information ratio scales with the square root
> of the number of independent bets, and twelve assets is a small number of bets. **The IC table
> is a screening tool, not evidence.**

In [ ]:
IC_HORIZONS = (21, 63, 252)


def add_forward_returns(frame, horizons):
    """Forward total return over each horizon, computed within each asset."""
    ordered_frame = frame.sort_values(["ticker", "m_date"])
    price = ordered_frame["m_close_dividend_and_split_adjusted"]
    grouped = price.groupby(ordered_frame["ticker"])
    return pandas.DataFrame(
        {f"forward_{horizon}d": (grouped.shift(-horizon) / price) - 1.0 for horizon in horizons},
        index=ordered_frame.index,
    )


def cross_sectional_ic(frame, signal_column, forward_column):
    """
    Mean per-date Spearman IC, its dispersion, and the implied information ratio.

    Both series are re-ranked inside each date and correlated there, so this is one cross-section
    at a time - never pooled across dates, which would let the sample's own time trend masquerade
    as predictive power.
    """
    usable = frame[["m_date", signal_column, forward_column]].dropna()
    if usable.empty:
        return {"ic": numpy.nan, "ic_std": numpy.nan, "ir": numpy.nan, "dates": 0}
    grouped = usable.groupby("m_date")
    ranked = pandas.DataFrame({
        "m_date": usable["m_date"],
        "signal": grouped[signal_column].rank(pct=True),
        "forward": grouped[forward_column].rank(pct=True),
    })
    by_date = ranked.groupby("m_date")
    signal_deviation = ranked["signal"] - by_date["signal"].transform("mean")
    forward_deviation = ranked["forward"] - by_date["forward"].transform("mean")
    products = (signal_deviation * forward_deviation).groupby(ranked["m_date"]).sum()
    signal_energy = (signal_deviation ** 2).groupby(ranked["m_date"]).sum()
    forward_energy = (forward_deviation ** 2).groupby(ranked["m_date"]).sum()
    denominator = numpy.sqrt(signal_energy * forward_energy)
    daily_ic = (products / denominator.where(denominator > 0)).dropna()
    return {
        "ic": daily_ic.mean(),
        "ic_std": daily_ic.std(),
        "ir": daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else numpy.nan,
        "dates": len(daily_ic),
    }


with_forward = panel.join(add_forward_returns(panel, IC_HORIZONS))
scopes = [("all securities", with_forward)]
if ELIGIBILITY_COLUMN is not None:
    eligible = with_forward[with_forward[ELIGIBILITY_COLUMN] == 1.0]
    scopes.append(("eligible pool", eligible))
    print(f"Whole panel  : {len(with_forward):,} rows")
    print(f"Eligible pool: {len(eligible):,} rows ({len(eligible) / len(with_forward):.0%})\n")

ic_rows = []
for scope_name, scope in scopes:
    for signal_column in CANDIDATE_RANK_COLUMNS:
        for horizon in IC_HORIZONS:
            ic_rows.append({
                "scope": scope_name,
                "signal": signal_column.replace("r_", "").replace("_rank", ""),
                "horizon": f"{horizon}d",
                **cross_sectional_ic(scope, signal_column, f"forward_{horizon}d"),
            })

ic_table = pandas.DataFrame(ic_rows)
print("Information coefficient:")
print(ic_table.pivot_table(index="signal", columns=["scope", "horizon"], values="ic")
      .round(4).to_string())
print("\nInformation ratio (IC / IC std):")
print(ic_table.pivot_table(index="signal", columns=["scope", "horizon"], values="ir")
      .round(3).to_string())

ic_path = CHART_DIR.parent / "signal_information_coefficients.csv"
ic_table.to_csv(ic_path, index=False)
print(f"\nWritten: {ic_path.relative_to(REPO_ROOT)}")

decisive_scope = scopes[-1][0]
scope_ic = ic_table[ic_table["scope"] == decisive_scope]
signals_ordered = (
    scope_ic.groupby("signal")["ic"].apply(lambda values: values.abs().max())
    .sort_values(ascending=False).index
)
figure, axes = matplotlib.pyplot.subplots(figsize=(10, 0.9 * len(signals_ordered) + 1.8))
positions = numpy.arange(len(signals_ordered))
for offset, horizon in zip((-0.26, 0.0, 0.26), (f"{h}d" for h in IC_HORIZONS)):
    axes.barh(
        positions + offset,
        [scope_ic.loc[(scope_ic["signal"] == signal) & (scope_ic["horizon"] == horizon),
                      "ic"].squeeze() for signal in signals_ordered],
        height=0.26, label=horizon,
    )
axes.set_yticks(positions)
axes.set_yticklabels(signals_ordered)
axes.axvline(0, color=INK, linewidth=1.2)
axes.legend(frameon=False, fontsize=9, title="forward horizon", ncol=3)
style_axes(axes, f"Information coefficient, {decisive_scope}",
           subtitle="positive means a high rank predicts a high forward return;"
                    " negative means the feature works inverted",
           xlabel="mean per-date Spearman IC")
axes.spines["left"].set_visible(False)
save(figure, "signal_information_coefficient.png")

---

## 9 · Handoff

| Output | Consumed by |
| --- | --- |
| `Data/Refinery/Time_Series/` | **`Experiments/` — read this one** |
| `Data/Analyzer/signal_information_coefficients.csv` | feature selection in every experiment |
| `Data/Analyzer/Charts/` | `FINDINGS_N.md` |

In [ ]:
best = scope_ic.reindex(scope_ic["ic"].abs().sort_values(ascending=False).index).iloc[0]

metrics = ["securities", "priced from", "trading days", "candidate features"]
values = [
    f"{panel['ticker'].nunique()}",
    f"{panel['m_date'].min().date()}",
    f"{panel['m_date'].nunique():,}",
    f"{len(CANDIDATE_RANK_COLUMNS)}",
]
# --- example: begin ---
metrics += [
    "signalled from", "median regime spell", "switches per asset per year", "mean breadth",
    "good regime earns more", "good regime is calmer", "look-ahead worth",
]
values += [
    f"{signalled['m_date'].min().date()}",
    f"{numpy.median(spell_lengths):.0f} trading days",
    f"{persistence['switches_per_year'].mean():.1f}",
    f"{breadth.mean():.0%}",
    f"{return_wins}/{len(separation)} assets ({separation['return_spread'].mean():+.1%})",
    f"{volatility_wins}/{len(separation)} assets ({separation['volatility_drop'].mean():+.1%})",
    f"{lookahead['cost_of_honesty'].mean():.1%} a year",
]
# --- example: end ---
metrics += [f"strongest feature ({decisive_scope})"]
values += [f"{best['signal']} @ {best['horizon']} (IC {best['ic']:+.4f})"]

print(pandas.DataFrame({"metric": metrics, "value": values}).to_string(index=False))
print(f"\nCharts: {CHART_DIR.relative_to(REPO_ROOT)}")

## What to write in the blueprint before running an experiment

The predictions this notebook licenses. Put them in `BLUEPRINT_N.md` **before** the backtest, so
the engine gets to confirm or refute something that was stated first.

| # | Prediction | Where it comes from |
| --- | --- | --- |
| 1 | Holding only good-regime assets will **cut volatility and drawdown** rather than raise return. | Section 4: the volatility split holds across nearly every asset, the return split does not. |
| 2 | The Sharpe improvement, if any, will come through the denominator. | The same. |
| 3 | Any variant that reports a large return gain from the regime signal alone should be **suspected of look-ahead** before it is believed. | Section 5 puts a number on what look-ahead is worth here. |
| 4 | The conclusion should not move much as the jump penalty changes. | Section 6, if the curve is flat in sign. |

## Open items

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **The model identifies a regime; it does not forecast one.** The source paper adds a supervised layer that predicts tomorrow's regime from today's features. That is a strategy decision, so it belongs in an experiment rather than in the Data stage. |
| 2 | **Twelve assets is a small cross-section.** Every cross-sectional statistic here is noisy, and the fundamental law says an information ratio scales with the square root of the number of independent bets. Widening the universe is the cheapest way to raise the ceiling. |
| 3 | **Curator output is not reproducible across download dates.** Dividend adjustment is computed from the present, so a re-pull rebases every adjusted column and moves every number here in the third decimal. |
| 4 | **The IC is measured on ranks, not on a traded portfolio.** A feature with a good IC can still lose money after costs. Read section 8 together with the turnover in section 3.1. |